# 05 - VAE on EMG (regressor head)

Same backbone as 03 (EMG 1D CNN VAE) but with a scalar regressor head instead of 5-way classifier. Matches Rahman et al. 2025 task framing.

Anti-collapse measures: z-scored target, class-balanced batch sampler. Reports MAE/RMSE in kg + nearest-class accuracy + directional bias by sex.

Outputs to `VAE/Results/vae_emg_reg/`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    mean_absolute_error, mean_squared_error, confusion_matrix
)

ROOT = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project")
TENSOR_PATH = ROOT / "VAE" / "Tensors" / "lifts_emg.npz"
OUT_DIR = ROOT / "VAE" / "Results" / "vae_emg_reg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Hyperparameters tuned to reduce regression-to-mean:
#   LATENT_DIM 16 -> 32   (more latent capacity for weight-discriminating features)
#   BETA       1.0 -> 0.1 (relax KL so mu values can spread along class-discriminative axes)
#   LAMBDA_REG 1.0 -> 10  (regression term dominates over recon/KL; mean-prediction is now costly)
LATENT_DIM = 32
BETA = 0.1
LAMBDA_REG = 10.0
BATCH_SIZE = 64
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-5
LOAD_BINS = np.array([2.3, 4.5, 6.8, 9.1, 11.3])

Device: cpu


In [2]:
# ---- load + drop NaN lifts (defensive; EMG had none in 03) ----
data = np.load(TENSOR_PATH, allow_pickle=True)
X = data["X"].astype(np.float32)
participant = data["participant"]
sex = data["sex"]
box_mass = data["box_mass"].astype(np.float32)
channel_names = data["channel_names"]

nan_lift = np.isnan(X).any(axis=(1, 2))
print(f"Dropping {int(nan_lift.sum())} / {len(X)} lifts with NaN")
keep = ~nan_lift
X = X[keep]; participant = participant[keep]; sex = sex[keep]; box_mass = box_mass[keep]

def mass_to_class(m): return int(np.argmin(np.abs(LOAD_BINS - m)))
y_cls = np.array([mass_to_class(m) for m in box_mass], dtype=np.int64)

print("X shape:", X.shape, "subjects:", len(np.unique(participant)))
print("Class dist:", dict(zip(*np.unique(y_cls, return_counts=True))))

Dropping 0 / 4750 lifts with NaN
X shape: (4750, 400, 8) subjects: 40
Class dist: {0: 947, 1: 960, 2: 947, 3: 948, 4: 948}


In [3]:
# ---- split (same scheme as 03) ----
subjects = np.unique(participant)
subj_sex = {p: sex[participant == p][0] for p in subjects}
rng = np.random.default_rng(SEED)
test_subjects = []
for s_label in np.unique(list(subj_sex.values())):
    subs_s = np.array([p for p, s in subj_sex.items() if s == s_label])
    rng.shuffle(subs_s)
    n_test = max(1, int(round(0.2 * len(subs_s))))
    test_subjects.extend(subs_s[:n_test].tolist())
test_subjects = set(test_subjects)
train_subjects = set(subjects) - test_subjects
train_mask = np.array([p in train_subjects for p in participant])
test_mask = ~train_mask
print(f"Train subjects: {len(train_subjects)} lifts: {train_mask.sum()}  |  Test subjects: {len(test_subjects)} lifts: {test_mask.sum()}")

Train subjects: 32 lifts: 3791  |  Test subjects: 8 lifts: 959


In [4]:
X_train_raw = X[train_mask]
ch_mean = X_train_raw.reshape(-1, X.shape[-1]).mean(axis=0)
ch_std = X_train_raw.reshape(-1, X.shape[-1]).std(axis=0) + 1e-6
Xn = (X - ch_mean) / ch_std

y_mean = float(box_mass[train_mask].mean())
y_std = float(box_mass[train_mask].std() + 1e-6)
y_norm = (box_mass - y_mean) / y_std
print(f"y_mean={y_mean:.3f}  y_std={y_std:.3f}")

X_train = torch.from_numpy(Xn[train_mask]).transpose(1, 2)
X_test = torch.from_numpy(Xn[test_mask]).transpose(1, 2)
y_train = torch.from_numpy(y_norm[train_mask]).float()
y_test = torch.from_numpy(y_norm[test_mask]).float()
cls_train = y_cls[train_mask]

class_counts = np.bincount(cls_train, minlength=len(LOAD_BINS))
sample_w = 1.0 / class_counts[cls_train]
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_w).double(),
                                num_samples=len(sample_w), replacement=True)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, sampler=sampler)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)
print("X_train:", X_train.shape, "X_test:", X_test.shape)

y_mean=6.794  y_std=3.193
X_train: torch.Size([3791, 8, 400]) X_test: torch.Size([959, 8, 400])


In [5]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(in_c, out_c, k, s, p), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, op):
        super().__init__()
        self.net = nn.Sequential(nn.ConvTranspose1d(in_c, out_c, k, s, p, output_padding=op), nn.BatchNorm1d(out_c), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class VAE(nn.Module):
    def __init__(self, n_channels, seq_len=400, latent_dim=16):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBlock(n_channels, 32, 7, 2, 3),
            ConvBlock(32, 64, 5, 2, 2),
            ConvBlock(64, 128, 3, 2, 1),
        )
        self.enc_out_len = seq_len // 8
        self.flat_dim = 128 * self.enc_out_len
        self.fc_mu = nn.Linear(self.flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.flat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, self.flat_dim)
        self.dec = nn.Sequential(
            DeconvBlock(128, 64, 3, 2, 1, 1),
            DeconvBlock(64, 32, 5, 2, 2, 1),
            nn.ConvTranspose1d(32, n_channels, 7, 2, 3, output_padding=1),
        )
        self.reg = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(inplace=True), nn.Linear(64, 1))

    def encode(self, x):
        h = self.enc(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparam(self, mu, logvar):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
    def decode(self, z):
        return self.dec(self.fc_dec(z).view(-1, 128, self.enc_out_len))
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        return self.decode(z), mu, logvar, z, self.reg(z).squeeze(-1)

model = VAE(n_channels=X.shape[-1], seq_len=X.shape[1], latent_dim=LATENT_DIM).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 697,225


In [6]:
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
history = []

def vae_losses(x, x_rec, mu, logvar, pred, y):
    recon = F.mse_loss(x_rec, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    reg = F.mse_loss(pred, y)
    return recon, kl, reg

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = {"recon": 0.0, "kl": 0.0, "reg": 0.0, "n": 0}
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        x_rec, mu, logvar, z, pred = model(xb)
        recon, kl, reg = vae_losses(xb, x_rec, mu, logvar, pred, yb)
        loss = recon + BETA * kl + LAMBDA_REG * reg
        loss.backward(); opt.step()
        bs = xb.size(0)
        tr["recon"] += recon.item()*bs; tr["kl"] += kl.item()*bs; tr["reg"] += reg.item()*bs; tr["n"] += bs

    model.eval()
    te = {"recon": 0.0, "kl": 0.0, "reg": 0.0, "n": 0}
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            x_rec, mu, logvar, z, pred = model(xb)
            recon, kl, reg = vae_losses(xb, x_rec, mu, logvar, pred, yb)
            bs = xb.size(0)
            te["recon"] += recon.item()*bs; te["kl"] += kl.item()*bs; te["reg"] += reg.item()*bs; te["n"] += bs

    row = {"epoch": epoch,
           "train_recon": tr["recon"]/tr["n"], "train_kl": tr["kl"]/tr["n"], "train_reg": tr["reg"]/tr["n"],
           "test_recon":  te["recon"]/te["n"], "test_kl":  te["kl"]/te["n"], "test_reg":  te["reg"]/te["n"]}
    history.append(row)
    if epoch % 5 == 0 or epoch == 1:
        print(f"ep{epoch:3d} | tr_rec {row['train_recon']:.3f} kl {row['train_kl']:.3f} reg {row['train_reg']:.3f} "
              f"| te_rec {row['test_recon']:.3f} kl {row['test_kl']:.3f} reg {row['test_reg']:.3f}")

pd.DataFrame(history).to_csv(OUT_DIR / "history.csv", index=False)

ep  1 | tr_rec 0.965 kl 20.402 reg 0.671 | te_rec 0.717 kl 4.047 reg 0.974
ep  5 | tr_rec 0.760 kl 3.323 reg 0.404 | te_rec 0.595 kl 3.105 reg 0.783
ep 10 | tr_rec 0.629 kl 3.625 reg 0.203 | te_rec 0.610 kl 3.340 reg 1.124
ep 15 | tr_rec 0.592 kl 3.175 reg 0.120 | te_rec 0.546 kl 3.042 reg 0.908
ep 20 | tr_rec 0.550 kl 2.729 reg 0.065 | te_rec 0.518 kl 2.759 reg 0.961
ep 25 | tr_rec 0.514 kl 2.702 reg 0.045 | te_rec 0.496 kl 2.565 reg 0.890
ep 30 | tr_rec 0.517 kl 2.656 reg 0.049 | te_rec 0.493 kl 2.503 reg 0.951
ep 35 | tr_rec 0.461 kl 2.431 reg 0.035 | te_rec 0.495 kl 2.281 reg 0.865
ep 40 | tr_rec 0.437 kl 2.354 reg 0.034 | te_rec 0.487 kl 2.254 reg 0.914
ep 45 | tr_rec 0.432 kl 2.351 reg 0.037 | te_rec 0.490 kl 2.129 reg 0.863
ep 50 | tr_rec 0.419 kl 2.351 reg 0.035 | te_rec 0.491 kl 2.235 reg 0.891
ep 55 | tr_rec 0.407 kl 2.351 reg 0.035 | te_rec 0.488 kl 2.247 reg 0.873
ep 60 | tr_rec 0.396 kl 2.156 reg 0.025 | te_rec 0.484 kl 1.977 reg 0.888


In [7]:
model.eval()
all_mu, all_pred_norm = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        mu, _ = model.encode(xb)
        pred = model.reg(mu).squeeze(-1)
        all_mu.append(mu.cpu().numpy())
        all_pred_norm.append(pred.cpu().numpy())
mu_test = np.concatenate(all_mu)
pred_norm = np.concatenate(all_pred_norm)

mass_pred = pred_norm * y_std + y_mean
mass_true = box_mass[test_mask]
y_true_cls = np.array([mass_to_class(m) for m in mass_true])
y_pred_cls = np.array([mass_to_class(m) for m in mass_pred])

mae = mean_absolute_error(mass_true, mass_pred)
rmse = np.sqrt(mean_squared_error(mass_true, mass_pred))
acc = accuracy_score(y_true_cls, y_pred_cls)
bal_acc = balanced_accuracy_score(y_true_cls, y_pred_cls)

print(f"Test  MAE={mae:.3f} kg  RMSE={rmse:.3f} kg  nearest-class acc={acc:.3f}  bal_acc={bal_acc:.3f}")
print("\nPrediction distribution (kg):")
print(pd.Series(mass_pred).describe().round(3).to_string())
print("\nMean predicted kg by true class (regression-to-mean check):")
for k, m in enumerate(LOAD_BINS):
    sel = y_true_cls == k
    if sel.any():
        print(f"  true={m:5.2f} kg  n={int(sel.sum()):3d}  pred_mean={mass_pred[sel].mean():.3f}  pred_std={mass_pred[sel].std():.3f}")

Test  MAE=2.369 kg  RMSE=3.006 kg  nearest-class acc=0.331  bal_acc=0.331

Prediction distribution (kg):
count    959.000
mean       6.661
std        2.810
min        0.474
25%        4.461
50%        6.643
75%        8.940
max       13.718

Mean predicted kg by true class (regression-to-mean check):
  true= 2.30 kg  n=192  pred_mean=4.214  pred_std=2.243
  true= 4.50 kg  n=192  pred_mean=5.950  pred_std=2.429
  true= 6.80 kg  n=191  pred_mean=7.030  pred_std=2.524
  true= 9.10 kg  n=192  pred_mean=7.792  pred_std=2.488
  true=11.30 kg  n=192  pred_mean=8.321  pred_std=2.295


In [8]:
test_sex = sex[test_mask]
signed_err = mass_pred - mass_true
abs_err = np.abs(signed_err)
fair_rows = []
for s_label in np.unique(test_sex):
    m = test_sex == s_label
    fair_rows.append({
        "sex": s_label, "n": int(m.sum()),
        "mae_kg": float(abs_err[m].mean()),
        "rmse_kg": float(np.sqrt((signed_err[m] ** 2).mean())),
        "mean_signed_err_kg": float(signed_err[m].mean()),
        "over_rate": float((signed_err[m] > 0).mean()),
        "under_rate": float((signed_err[m] < 0).mean()),
        "nearest_acc": accuracy_score(y_true_cls[m], y_pred_cls[m]),
    })
fair_df = pd.DataFrame(fair_rows)
print(fair_df.to_string(index=False))

torch.save({"model_state": model.state_dict(), "ch_mean": ch_mean, "ch_std": ch_std,
            "y_mean": y_mean, "y_std": y_std,
            "latent_dim": LATENT_DIM, "seq_len": X.shape[1], "n_channels": X.shape[-1],
            "channel_names": list(map(str, channel_names))}, OUT_DIR / "vae_emg_reg.pt")
np.savez(OUT_DIR / "test_outputs.npz", mu=mu_test,
         mass_true=mass_true, mass_pred=mass_pred,
         y_true_cls=y_true_cls, y_pred_cls=y_pred_cls,
         participant=participant[test_mask], sex=test_sex)
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump({"mae_kg": float(mae), "rmse_kg": float(rmse),
               "nearest_acc": float(acc), "nearest_bal_acc": float(bal_acc),
               "confusion_matrix": confusion_matrix(y_true_cls, y_pred_cls).tolist(),
               "fairness": fair_rows,
               "config": {"latent_dim": LATENT_DIM, "beta": BETA, "lambda_reg": LAMBDA_REG,
                          "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "seed": SEED}}, f, indent=2)
fair_df.to_csv(OUT_DIR / "fairness.csv", index=False)
print("Saved to:", OUT_DIR)

   sex   n   mae_kg  rmse_kg  mean_signed_err_kg  over_rate  under_rate  nearest_acc
Female 479 2.354458 2.912488            0.002202   0.507307    0.492693     0.323591
  Male 480 2.384129 3.096214           -0.279658   0.470833    0.529167     0.337500
Saved to: C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\VAE\Results\vae_emg_reg
